# 📝 W2: 교안 & 학습지 자동화
**hexa-4 — Week 2** | ⏱️ 60분 | 🎯 교안+평가+워크시트 자동 생성

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
!pip install -q google-generativeai pandas matplotlib
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 수업 정보 입력 (✏️)

In [ ]:
INFO={'subject':'✏️ [과목]','grade':'✏️ [학년]','unit':'✏️ [단원명]','duration':45}
print('✅',INFO['unit'])


## Step 2: 교안 자동 생성 (도입/전개/정리)

In [ ]:
prompt=f"""교육 전문가로서 수업지도안을 작성하세요.
과목: {INFO['subject']} / 학년: {INFO['grade']} / 단원: {INFO['unit']} / 수업시간: {INFO['duration']}분
형식:
## 1. 도입({INFO['duration']//5}분)
- 학습목표:
- 도입활동:
## 2. 전개({INFO['duration']//5*3}분)
- 핵심내용 3가지:
- 활동:
## 3. 정리({INFO['duration']//5}분)
- 정리질문:
- 과제:"""
lesson=model.generate_content(prompt)
print(lesson.text)


## Step 3: 형성평가 5문제 자동 생성

In [ ]:
p2=f"""{INFO['unit']} 단원의 형성평가 문제를 만드세요.
객관식 3문제 (4지선다, 정답표시) + 주관식 2문제. 마크다운 형식."""
quiz=model.generate_content(p2); print(quiz.text)


## Step 4: 학습지(워크시트) 생성

In [ ]:
p3=f"""학생 작성용 워크시트를 만드세요. (빈칸 ___ 포함)
단원: {INFO['unit']}
핵심 개념 3개 빈칸 + 정리 질문 2개. 마크다운."""
ws=model.generate_content(p3); print(ws.text)


## Step 5: ZIP 다운로드

In [ ]:
import zipfile
for name,content in [('교안.md',lesson.text),('평가.md',quiz.text),('학습지.md',ws.text)]:
    with open(name,'w',encoding='utf-8') as f: f.write(content)
with zipfile.ZipFile('lesson_package.zip','w') as z:
    for name in ['교안.md','평가.md','학습지.md']: z.write(name)
from google.colab import files; files.download('lesson_package.zip')
print('✅ 다운로드 완료')
